In [12]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import pandas as pd
from collections import defaultdict
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [20]:
class FOAFRecommender:
    def __init__(self, num_users, num_items, latent_dim=20, lr=0.01, lambda_reg=0.01):
        self.num_users = num_users
        self.num_items = num_items
        self.latent_dim = latent_dim
        self.lr = lr
        self.lambda_reg = lambda_reg

        self.P = np.random.randn(num_users, latent_dim) * 0.1
        self.Q = np.random.randn(num_items, latent_dim) * 0.1

        self.user_data = defaultdict(list)
        self.graph = nx.Graph()

    def add_interaction(self, user_id, item_id, rating):
        self.user_data[user_id].append((item_id, rating))

    def build_graph(self, edges):
        self.graph.add_edges_from(edges)

    def compute_gradient(self, user_id):
        user_vec = self.P[user_id]
        grad_p = np.zeros_like(user_vec)
        grad_Q = np.zeros_like(self.Q)

        interactions = self.user_data[user_id]
        if len(interactions) > 10:
            indices = np.random.choice(len(interactions), 10, replace=False)
            sampled = [interactions[i] for i in indices]
        else:
            sampled = interactions

        for item_id, rating in sampled:
            item_vec = self.Q[item_id]
            pred = np.dot(user_vec, item_vec)
            error = rating - pred

            grad_p += -error * item_vec + self.lambda_reg * user_vec
            grad_Q[item_id] += -error * user_vec + self.lambda_reg * item_vec

        return grad_p, grad_Q

    def train_user(self, user_id):
        grad_p, grad_Q = self.compute_gradient(user_id)
        self.P[user_id] -= self.lr * grad_p
        self.Q -= self.lr * grad_Q
        for neighbor in self.graph.neighbors(user_id):
            self.Q -= self.lr * grad_Q

    def train(self, epochs=5):
        for epoch in range(epochs):
            for user_id in range(self.num_users):
                self.train_user(user_id)

    def predict(self, user_id, item_id):
        return np.dot(self.P[user_id], self.Q[item_id])

    def evaluate_rmse(self, test_data):
        se = 0.0
        for user_id, item_id, true_rating in test_data:
            pred = self.predict(user_id, item_id)
            se += (true_rating - pred) ** 2
        return np.sqrt(se / len(test_data))

In [32]:
# === Load MovieLens 100K ===
data = pd.read_csv("u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
data["user_id"] -= 1
data["item_id"] -= 1

num_users = data["user_id"].nunique()
num_items = data["item_id"].nunique()

train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)
train_data = list(train_df[["user_id", "item_id", "rating"]].itertuples(index=False, name=None))
test_data = list(test_df[["user_id", "item_id", "rating"]].itertuples(index=False, name=None))


In [34]:
# Build recommender
model = FOAFRecommender(num_users=num_users, num_items=num_items, latent_dim=20, lr=0.01, lambda_reg=0.01)



In [36]:
for user_id, item_id, rating in train_data:
    model.add_interaction(user_id, item_id, rating)

edges = []
for user in range(num_users):
    neighbors = np.random.choice([u for u in range(num_users) if u != user], 5, replace=False)
    for n in neighbors:
        edges.append((user, n))
model.build_graph(edges)




In [38]:
# Train model
model.train(epochs=5)

In [39]:
# Evaluate RMSE
rmse = model.evaluate_rmse(test_data)
print(f"Test RMSE: {rmse:.4f}")


Test RMSE: 1.6132
